# SAFE binary v3 — AI-Hub 포함 / PAN12 제외 오탐 개선 실험

목표: 누수 없는 `시드 + 합성×1 + AI-Hub train`으로 이진 모델을 재학습하고, 실데이터·3a 그루밍·warm-normal 회귀셋을 함께 평가한다. PAN12는 원인 분리를 위해 제외한다. **승인 게이트를 통과하기 전에는 HF에 업로드하지 않는다.**

In [ ]:
# 1. GPU 확인
import subprocess, sys
subprocess.run(['nvidia-smi', '-L'], check=True)
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 필요합니다.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. 저장소 준비 및 최신 feature 브랜치 동기화
from pathlib import Path
REPO = Path('/content/thisabled-ai')
BRANCH = 'feature/grooming-augmentation'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/threeGuineas/thisabled-ai.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True, text=True, capture_output=True).stdout)

In [ ]:
# 3. 의존성 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
print('dependencies ready')

In [ ]:
# 4. data_bundle.zip 업로드 (VS Code/Jupyter 호환)
import io, json, pathlib, zipfile
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(accept='.zip', multiple=False, description='data_bundle.zip 선택')
def on_upload(change):
    value = uploader.value
    if not value: return
    item = next(iter(value.values())) if isinstance(value, dict) else value[0]
    with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
        required = {
            'data/synthetic/pan12_translated.jsonl',
            'data/eval/aihub_train.jsonl',
            'data/eval/aihub_real_holdout.jsonl',
            'data/eval/beep_real_holdout.jsonl',
        }
        missing = sorted(required - set(z.namelist()))
        assert not missing, f'ZIP 필수 파일 누락: {missing}'
        z.extractall(REPO)
    pan = [json.loads(x) for x in (REPO/'data/synthetic/pan12_translated.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
    print('업로드 완료:', len(pan), '건 / predator', sum(x.get('split_role') == 'predator' for x in pan), '건')
uploader.observe(on_upload, names='value')
display(uploader)

In [ ]:
# 5. raw 다운로드 → 시드 split → 합성×1 + 누수 제거된 AI-Hub train 병합
steps = [
    [sys.executable, 'scripts/download_seed_datasets.py'],
    [sys.executable, 'scripts/build_processed_dataset.py'],
    [sys.executable, 'scripts/build_final_dataset.py', '--synth-repeat', '1', '--include-aihub-train'],
]
for cmd in steps:
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
# 6. 분포·누수 사전 검증
import re, pandas as pd
train = pd.read_parquet(REPO/'data/processed/train.parquet')
def norm(text): return re.sub(r'[^0-9a-z가-힣]+', '', str(text).lower())
holdout_norm = set()
for name in ['aihub_real_holdout.jsonl', 'beep_real_holdout.jsonl']:
    for line in (REPO/'data/eval'/name).open(encoding='utf-8'):
        holdout_norm.add(norm(json.loads(line)['text']))
overlap = int(train.text.map(norm).isin(holdout_norm).sum())
binary_dist = (train.label.astype(int) > 0).astype(int).value_counts().sort_index().to_dict()
print('총건수:', len(train))
print('binary:', binary_dist)
print('source:', train.source.value_counts().to_dict())
print('train↔holdout 중복:', overlap)
assert overlap == 0
assert 'aihub_real' in set(train.source)
assert binary_dist.get(0, 0) > 0 and binary_dist.get(1, 0) > 0

In [ ]:
# 7. PAN12 제외 임시 config 생성 및 학습
import shutil, yaml
base = REPO/'configs/module1_binary.yaml'
cfg = yaml.safe_load(base.read_text(encoding='utf-8'))
cfg['data']['extra_caution_jsonl'] = []
cfg['model']['checkpoint_dir'] = 'models/checkpoints/module1_binary_aihub_no_pan'
cfg['paths']['checkpoint_dir'] = 'models/checkpoints/module1_binary_aihub_no_pan'
assert cfg['training']['seed'] == 42 and cfg['model']['num_labels'] == 2
TEMP_CONFIG = Path('/content/module1_binary_aihub_no_pan.yaml')
TEMP_CONFIG.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
CKPT = REPO/'models/checkpoints/module1_binary_aihub_no_pan'
FORCE_RETRAIN = False  # 기존 성공 체크포인트가 있으면 재사용
if FORCE_RETRAIN and CKPT.exists(): shutil.rmtree(CKPT)
if not (CKPT/'model.safetensors').exists():
    subprocess.run([sys.executable, 'scripts/train_module1.py', '--config', str(TEMP_CONFIG)], cwd=REPO, check=True)
for name in ['config.json','model.safetensors','tokenizer.json','tokenizer_config.json','vocab.txt']:
    assert (CKPT/name).exists(), f'체크포인트 누락: {name}'
print('checkpoint:', CKPT)

In [ ]:
# 8. 실데이터 홀드아웃 + 3a 그루밍 평가
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, f1_score
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).cuda().eval()
assert model.config.num_labels == 2
def predict_prob(texts, batch=64):
    out=[]
    with torch.inference_mode():
        for i in range(0,len(texts),batch):
            enc=tok(texts[i:i+batch],padding=True,truncation=True,max_length=128,return_tensors='pt').to('cuda')
            out.extend(torch.softmax(model(**enc).logits,dim=-1)[:,1].cpu().tolist())
    return np.array(out)
real=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    real += [json.loads(x) for x in (REPO/'data/eval'/name).open(encoding='utf-8') if x.strip()]
y_real=np.array([int(int(x['label'])>0) for x in real]); p_real=predict_prob([x['text'] for x in real])
groom=[]
for split in ['val','test']:
    groom += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').open(encoding='utf-8') if x.strip()]
p_groom=predict_prob([x['text'] for x in groom])
REAL_RESULTS={}
for tau in [0.35,0.5,0.9]:
    pred=(p_real>=tau).astype(int); tn,fp,fn,tp=confusion_matrix(y_real,pred,labels=[0,1]).ravel()
    row={'recall':tp/(tp+fn),'specificity':tn/(tn+fp),'fpr':fp/(tn+fp),'macro_f1':f1_score(y_real,pred,average='macro'),'cm':[[int(tn),int(fp)],[int(fn),int(tp)]],'grooming_recall':float((p_groom>=tau).mean())}
    REAL_RESULTS[str(tau)]=row; print('tau',tau,row)

In [ ]:
# 9. 고정 warm-normal 회귀셋 + 위험 대조셋 평가 (학습 사용 금지)
normal_cases = [
'오늘 날씨 진짜 좋다! 산책 갈래?','과제 다 했어? 나 아직 반도 못했어 ㅠㅠ','내일 몇 시에 만날까? 카페에서 보자','어제 영화 진짜 재밌었어 너도 봐봐','밥은 먹었어? 오늘 하루 어땠어','이번 주말에 같이 등산 가자 날씨 좋대','역까지 데려다줄게, 짐 많아 보이는데 도와줄까?','요즘 잘 지내? 얼굴 본 지 오래됐다','그 카페 신메뉴 나왔대 다음에 같이 가보자','시험 끝나고 뭐 할 거야? 놀러 가자','생일 축하해! 선물 뭐 갖고 싶어?','네 발표 진짜 잘하더라 대단해','혹시 필기 좀 빌려줄 수 있어? 어제 결석했어','우리 팀플 자료 정리 내가 할게 걱정 마','몸 좀 괜찮아? 무리하지 말고 푹 쉬어',
'오늘 점심 뭐 먹을까?','버스가 조금 늦는대','사진 잘 나왔다 보내줄게','이번 주말에 비 온다더라','숙제 끝나면 게임하자','회의 자료 메일로 보냈어요','오늘 영화 정말 재미있었어','집에 도착하면 연락해','병원 갈 때 같이 가줄게','길 찾기 어려우면 알려줘','필요한 자료 있으면 공유할게','휠체어 경사로는 건물 뒤쪽에 있어요','약 먹는 시간 잊지 않았지?','우산 없으면 같이 쓰고 가자','천천히 와도 돼 기다리고 있을게'
]
risk_cases = [
'우리 둘만 아는 비밀로 하고 부모님께 말하지 마','너는 나만 믿으면 돼 다른 친구들은 필요 없어','혼자 있을 때 사진 찍어서 보내줘','집 주소 알려주면 몰래 데리러 갈게','선생님한테는 말하지 말고 우리끼리 만나자','개인적인 영상 보내주면 선물 사줄게','부모님 없을 때 영상통화하자','다른 사람에게 들키면 안 되니까 대화 기록 지워','계좌번호와 비밀번호 보내면 돈을 입금해줄게','지금 돈 안 보내면 네 사진을 공개하겠다','주소 다 알고 있으니 오늘 찾아간다','죽여버릴 테니 밖으로 나와','인증번호 알려주면 지원금을 받을 수 있어요','내 말 안 들으면 가족에게 전부 알릴 거야','수수료를 먼저 보내야 환급받을 수 있습니다','네 사진을 단톡방에 뿌리기 전에 답해','오늘 밤 집 앞에서 기다릴 테니 도망가지 마','살 가치도 없으니 그냥 죽어'
]
p_normal=predict_prob(normal_cases); p_risk=predict_prob(risk_cases)
REGRESSION_RESULTS={}
for tau in [0.35,0.5,0.9]:
    specificity=float((p_normal<tau).mean()); recall=float((p_risk>=tau).mean())
    REGRESSION_RESULTS[str(tau)]={'risk_recall':recall,'specificity':specificity,'fpr':1-specificity,'normal_fp':int((p_normal>=tau).sum()),'normal_n':len(normal_cases)}
    print('tau',tau,REGRESSION_RESULTS[str(tau)])
print('tau=0.5 false positives:')
for text,p in zip(normal_cases,p_normal):
    if p>=0.5: print(f'FP p={p:.4f} | {text}')
print('tau=0.5 false negatives:')
for text,p in zip(risk_cases,p_risk):
    if p<0.5: print(f'FN p={p:.4f} | {text}')

In [ ]:
# 10. 승인 게이트 및 결과 저장
RESULT={
    'experiment':'binary_aihub_no_pan',
    'seed':42,
    'train_n':int(len(train)),
    'train_binary_distribution':{str(k):int(v) for k,v in binary_dist.items()},
    'real_holdout':REAL_RESULTS,
    'regression':REGRESSION_RESULTS,
}
adult_real=REAL_RESULTS['0.5']; adult_reg=REGRESSION_RESULTS['0.5']
PASS=bool(adult_real['recall']>=0.80 and adult_reg['risk_recall']>=0.80 and adult_reg['specificity']>=0.90)
RESULT['pass']=PASS
out=REPO/'reports/validation_reports/module1_binary/aihub_no_pan_eval.json'
out.parent.mkdir(parents=True,exist_ok=True); out.write_text(json.dumps(RESULT,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(RESULT,ensure_ascii=False,indent=2))
print('APPROVED' if PASS else 'REJECTED — HF 업로드 금지')

In [ ]:
# 11. 승인 모델만 HF 업로드 — 기본 비활성
UPLOAD_TO_HF=False
if UPLOAD_TO_HF:
    assert PASS, '승인 게이트 실패 모델은 업로드할 수 없습니다.'
    from getpass import getpass
    from huggingface_hub import HfApi, login
    token=getpass('HF write token: '); login(token=token,add_to_git_credential=False); del token
    commit=HfApi().upload_folder(repo_id='soyuncj/thisabled-safety-kcelectra',repo_type='model',folder_path=str(CKPT),commit_message='retrain: binary AI-Hub in-domain, PAN excluded, leak-free',ignore_patterns=['checkpoint-*','checkpoint-*/*','optimizer*','scheduler*','trainer_state*','rng_state*','training_args*'])
    print('HF_COMMIT_URL:',commit.commit_url)
else:
    print('업로드 비활성 — 평가 결과를 먼저 검토하세요.')